# 10 — Special Matrices
## Identity, Diagonal, Symmetric & Positive Definite

> **Prerequisites:** Notebooks 01–09  
> **Difficulty:** ⭐⭐☆☆☆ Beginner-Intermediate  
> **Running example:** Covariance matrices, network weights, regularization

---

## Why "special" matrices?

Certain matrix structures appear over and over in ML. Recognizing them lets you:
- Use **faster, specialized algorithms** instead of general ones
- **Predict mathematical properties** without computing them
- Understand **what a matrix represents** from its structure alone

This notebook covers the five most important ones:

| Type | Defining property | ML appearance |
|---|---|---|
| **Identity** | I × A = A × I = A | Initialization, regularization |
| **Diagonal** | Non-zero only on diagonal | Feature scaling, SVD's Σ |
| **Symmetric** | A = Aᵀ | Covariance, kernel, Hessian |
| **Positive definite** | All eigenvalues > 0 | Valid covariance, convex loss |
| **Orthogonal** | QᵀQ = I | U, V in SVD; rotations |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as scipy_linalg

np.random.seed(42)

---
## 1. Identity Matrix — The Number 1 for Matrices

### What is it?
The identity matrix **I** has 1s on the diagonal and 0s everywhere else.

> **AI = IA = A** for any compatible matrix A

Exactly like multiplying a number by 1.

### In ML
- **Ridge regression**: adds λI to XᵀX → makes it invertible and adds regularization
- **Residual connections**: if the skip = identity, the network learns `F(x) + x`
- **Covariance**: isotropic Gaussian has covariance = σ²I (all directions equal)

In [ ]:
# ─────────────────────────────────────────────────────────
# IDENTITY MATRIX
# ─────────────────────────────────────────────────────────

print("=== Identity Matrix ===")
print()

I3 = np.eye(3)   # 3×3 identity matrix
I4 = np.eye(4)   # 4×4 identity matrix
print(f"I₃ =\n{I3}")
print()

# Verify: AI = IA = A
A = np.array([[2., 3.], [4., 5.]])
print(f"A =\n{A}")
print(f"A @ I₂ = A? {np.allclose(A @ np.eye(2), A)}")
print(f"I₂ @ A = A? {np.allclose(np.eye(2) @ A, A)}")
print()

print("=== Ridge regularization: adding λI to make XᵀX invertible ===")
np.random.seed(42)
X = np.random.randn(5, 5)
X_degenerate = np.hstack([X, X[:, :2]])  # add duplicate columns → singular

XtX = X_degenerate.T @ X_degenerate
lambda_ridge = 0.1
XtX_regularized = XtX + lambda_ridge * np.eye(XtX.shape[0])

print(f"det(XᵀX) without regularization: {np.linalg.det(XtX):.4e}")
print(f"det(XᵀX + 0.1I) with regularization:   {np.linalg.det(XtX_regularized):.4e}")
print(f"Without λI: singular? {abs(np.linalg.det(XtX)) < 1e-10}")
print(f"With λI: invertible? {abs(np.linalg.det(XtX_regularized)) > 1e-10}")
print("→ Adding λI guarantees invertibility (ridge regression trick!)")

---
## 2. Diagonal Matrix — Efficient Scaling

### What is it?
Non-zero values only on the main diagonal.

```
D = [[d₁,  0,  0],
     [ 0, d₂,  0],
     [ 0,  0, d₃]]
```

### Why it's special (and efficient)
- **D × v** just scales each element: result[i] = d[i] × v[i] — O(n) instead of O(n²)
- **D⁻¹** = diagonal(1/d₁, 1/d₂, ...) — just invert the diagonal entries
- **Eigenvalues = diagonal entries** (it's already diagonalized)

### In ML
- **Σ in SVD**: singular value matrix is diagonal
- **Λ in eigendecomposition**: eigenvalue matrix is diagonal
- **Feature scaling matrix**: each feature divided by its std = diagonal multiply

In [ ]:
# ─────────────────────────────────────────────────────────
# DIAGONAL MATRIX
# ─────────────────────────────────────────────────────────

print("=== Diagonal Matrix ===")
print()

D = np.diag([3., 2., 5., 1.])   # create diagonal matrix from a list
print(f"D =\n{D}")
print()
print(f"Extract diagonal entries: {np.diag(D)}")
print(f"Eigenvalues of D:         {np.linalg.eigvalsh(D)}  (= the diagonal entries!)")
print(f"det(D) = {np.linalg.det(D):.1f}  (= 3×2×5×1 = product of diagonals)")
print()

# Efficient inverse
D_inv = np.diag(1 / np.diag(D))   # just flip each diagonal entry
print(f"D⁻¹ =\n{D_inv}")
print(f"D @ D⁻¹ = I? {np.allclose(D @ D_inv, np.eye(4))}")
print()

print("=== Feature scaling as diagonal multiply ===")
X = np.array([[1500., 3., 10.],
              [2100., 4.,  5.],
              [1200., 2., 20.]])
stds = X.std(axis=0)
print(f"Feature stds: {stds}  (very different scales!)")

# Scale matrix: divide each feature by its std
S_inv = np.diag(1 / stds)  # diagonal matrix with 1/std values
X_scaled = X @ S_inv        # equivalent to dividing each column by its std
print(f"After diagonal scaling, stds: {X_scaled.std(axis=0).round(4)}")

---
## 3. Symmetric Matrix — Mirror Image

### What is it?
A matrix where **A[i, j] = A[j, i]** for all i, j. It's symmetric across the main diagonal.
> **A = Aᵀ**

### Guaranteed properties (Spectral Theorem)
1. All **eigenvalues are real** (not complex)
2. **Eigenvectors are orthogonal** — they form an orthonormal basis
3. Can always be written as **A = QΛQᵀ** with Q orthogonal

### In ML, symmetric matrices are everywhere
- **Covariance matrix**: Σ = (1/n) XᵀX — always symmetric
- **Gram matrix**: XᵀX and XXᵀ — always symmetric
- **Hessian**: matrix of second derivatives of the loss — always symmetric
- **Kernel matrix** in SVMs — always symmetric

In [ ]:
# ─────────────────────────────────────────────────────────
# SYMMETRIC MATRIX
# ─────────────────────────────────────────────────────────

print("=== Symmetric Matrix ===")
print()

# Manually defined symmetric matrix
S = np.array([[4., 2., 1.],
              [2., 5., 3.],
              [1., 3., 6.]])
print(f"S =\n{S}")
print(f"S == Sᵀ? {np.allclose(S, S.T)}")
print()

# All eigenvalues are real (use eigh for symmetric)
eigenvalues, eigenvectors = np.linalg.eigh(S)   # eigh: for symmetric/Hermitian matrices
print(f"Eigenvalues (all real): {eigenvalues.round(4)}")
print(f"Eigenvectors orthogonal? (VᵀV=I): {np.allclose(eigenvectors.T @ eigenvectors, np.eye(3))}")
print()

# Reconstruct: S = Q Λ Qᵀ
S_reconstructed = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T
print(f"Reconstruction S = QΛQᵀ error: {np.linalg.norm(S - S_reconstructed):.2e}")
print()

# Covariance matrix is always symmetric
print("=== Covariance matrix: always symmetric ===")
np.random.seed(42)
X = np.random.randn(100, 4)  # 100 samples × 4 features
X -= X.mean(axis=0)          # center
Cov = (X.T @ X) / (len(X) - 1)
print(f"Cov shape: {Cov.shape}")
print(f"Cov symmetric? {np.allclose(Cov, Cov.T)}")
print(f"Cov eigenvalues (all positive = valid covariance): {np.linalg.eigvalsh(Cov).round(4)}")

---
## 4. Positive Definite Matrix — The "Nice" Symmetric Matrix

### What is it?
A symmetric matrix A is **positive definite** (PD) if:
> **vᵀAv > 0  for every non-zero vector v**

In words: applying A to any vector v, then dotting with v, always gives a positive number.

### Equivalent tests (all mean the same thing)
1. All eigenvalues > 0
2. Cholesky decomposition A = LLᵀ exists (L is lower triangular)
3. All leading submatrices have positive determinant

### Positive semi-definite (PSD)
Same but eigenvalues ≥ 0 (allows zero). XᵀX is always PSD (not always PD).

### In ML
- **Covariance matrix**: if rank(X) = n_features, covariance is PD; otherwise PSD
- **Valid kernel**: Gram matrix must be PSD to be a valid kernel (Mercer's theorem)
- **Convex loss**: Hessian PD ↔ loss function is convex ↔ gradient descent will converge

In [ ]:
# ─────────────────────────────────────────────────────────
# POSITIVE DEFINITE MATRIX
# ─────────────────────────────────────────────────────────

print("=== Positive Definite Matrix ===")
print()

# PD matrix
A_pd = np.array([[5., 2.],
                 [2., 3.]])
vals_pd = np.linalg.eigvalsh(A_pd)
print(f"PD matrix A:")
print(A_pd)
print(f"Eigenvalues: {vals_pd}  (all positive → PD)")

# Verify vᵀAv > 0 for random vectors
np.random.seed(0)
random_vecs = np.random.randn(500, 2)
quadratic_forms = np.array([v @ A_pd @ v for v in random_vecs])
print(f"vᵀAv for 500 random v: min = {quadratic_forms.min():.4f}  (all positive? {(quadratic_forms > 0).all()})")
print()

# PSD but not PD
A_psd = np.array([[1., 1.],
                  [1., 1.]])
vals_psd = np.linalg.eigvalsh(A_psd)
print(f"PSD (but not PD) matrix:")
print(A_psd)
print(f"Eigenvalues: {vals_psd}  (one zero → PSD but not PD)")
print()

# NOT positive definite (has negative eigenvalue)
A_indef = np.array([[3., 5.],
                    [5., 2.]])
vals_indef = np.linalg.eigvalsh(A_indef)
print(f"Indefinite matrix:")
print(A_indef)
print(f"Eigenvalues: {vals_indef.round(4)}  (mixed sign → NOT PD, NOT a valid covariance!)")   

In [ ]:
# ─────────────────────────────────────────────────────────
# CHOLESKY DECOMPOSITION: A = LLᵀ
# A fast way to tell if a matrix is PD, and to sample from Gaussians
# ─────────────────────────────────────────────────────────

print("=== Cholesky Decomposition: A = LLᵀ ===")
print("Works only for positive definite matrices.")
print()

A = np.array([[4., 2., 1.],
              [2., 5., 3.],
              [1., 3., 6.]])

L = np.linalg.cholesky(A)   # L is lower triangular
print(f"A (PD matrix):")
print(A)
print(f"\nCholesky factor L (lower triangular):")
print(L.round(4))
print(f"\nL @ Lᵀ = A? {np.allclose(L @ L.T, A)}")
print()
print("=== Use case: sampling from a multivariate Gaussian ===")
print("If X ~ N(μ, Σ), then X = μ + L @ z  where z ~ N(0, I) and Σ = LLᵀ")
mean = np.array([10., 5., 3.])
Sigma = A  # use A as the covariance
L_cov = np.linalg.cholesky(Sigma)

# Sample 1000 points
z = np.random.randn(3, 1000)   # 1000 standard normal samples
samples = mean[:, None] + L_cov @ z   # transform to desired distribution
print(f"Sampled mean:  {samples.mean(axis=1).round(3)}  (should be {mean})")
print(f"Sampled cov diagonal: {np.var(samples, axis=1).round(3)}")
print(f"True cov diagonal:    {np.diag(Sigma)}")

---
## 5. Quick Visual: Structure of All Special Matrices

In [ ]:
# ─────────────────────────────────────────────────────────
# GALLERY: all special matrices as heatmaps
# ─────────────────────────────────────────────────────────

np.random.seed(42)
n = 6
A = np.random.randn(n, n)

special = [
    (np.eye(n),                        "Identity I"),
    (np.diag(np.arange(1., n+1)),      "Diagonal"),
    (A + A.T,                          "Symmetric (A+Aᵀ)"),
    (A.T @ A + 3*np.eye(n),            "Positive Definite (AᵀA+3I)"),
    (np.triu(A),                        "Upper Triangular"),
    (np.linalg.qr(A)[0],               "Orthogonal (Q from QR)"),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (M, title) in zip(axes.flat, special):
    # Determine color limits
    vmax = max(1.0, np.abs(M).max())
    im = ax.imshow(M, cmap='RdBu', vmin=-vmax, vmax=vmax)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=11, pad=8)
    ax.axis('off')
    # Add key property text
    props = {
        "Identity I": "AI = IA = A",
        "Diagonal": "Fast multiply & inverse",
        "Symmetric (A+Aᵀ)": "A = Aᵀ, real eigenvalues",
        "Positive Definite (AᵀA+3I)": "All eigenvalues > 0",
        "Upper Triangular": "Zeros below diagonal",
        "Orthogonal (Q from QR)": "QᵀQ = I, det = ±1",
    }
    ax.text(0.5, -0.05, props[title], transform=ax.transAxes,
            ha='center', fontsize=9, style='italic', color='gray')

plt.suptitle('Special Matrices: Structure Tells You Properties\n(red = positive, blue = negative)',
             fontsize=12)
plt.tight_layout()
plt.show()

---
## Summary

| Matrix type | Key property | Check in NumPy | ML use |
|---|---|---|---|
| Identity I | AI = A | `np.eye(n)` | Regularization: XᵀX + λI |
| Diagonal D | Non-zero only on diagonal | `np.diag(v)` | Singular values Σ, scaling |
| Symmetric A=Aᵀ | Mirror across diagonal | `np.allclose(A, A.T)` | Covariance, kernel, Hessian |
| Positive definite | All eigenvalues > 0 | `np.linalg.eigvalsh(A) > 0` | Valid covariance, convex loss |
| Orthogonal QᵀQ=I | Preserves geometry | `np.linalg.qr(A)[0]` | U,V in SVD; rotations |

**Next: Notebook 11 — Hands-On Lab** (put it all together!)